## Tratar dados os contatos

In [40]:
import pandas as pd
import json

In [41]:
lista_bairros = [
    "Brooklin",
    "Brooklin Novo",
    "Campo Belo",
    "Moema",
    "Vila Andrade"
]

In [60]:
lista_contatos = []

In [61]:
for bairro in lista_bairros:
    with open(f'dados/{bairro} - dados_anuncios.json') as f:
        lista_contatos.extend(json.loads(f.read()))

In [44]:
lista_contatos[0]

{'notification': 'success',
 'message': 'Dados carregados com sucesso',
 'error': False,
 'data': {'searchId': '1b4bf340-5980-4bd8-a751-aa0136642cc2',
  'totalElements': 1,
  'totalPages': 1,
  'people': [{'document': '82808937849',
    'name': 'Antonio Carlos Bressan',
    'birthDate': '',
    'type': 'PESSOA_FISICA',
    'addresses': [{'street': 'RUA INDIANA',
      'number': 1153,
      'extra': 'AP 61',
      'district': 'BROOKLIN PAULISTA',
      'city': 'SAO PAULO',
      'state': 'SP',
      'zipcode': 4562002,
      'latitude': 0,
      'longitude': 0,
      'iptuId': 'd66ada2b-ac48-4be0-a347-0dbe7afbab72',
      'zipCode': 4562002,
      'iptuMatch': True,
      'score': 1}]}]},
 'announce_id': 292463356}

In [11]:
len(lista_contatos)

28718

In [62]:
lista_contatos = [contato['data']['people'] for contato in lista_contatos]

In [64]:
lista_contatos = [contato for contatos in lista_contatos for contato in contatos]

In [70]:
lista_contatos[0]

{'document': '82808937849',
 'name': 'Antonio Carlos Bressan',
 'birthDate': '',
 'type': 'PESSOA_FISICA',
 'addresses': [{'street': 'RUA INDIANA',
   'number': 1153,
   'extra': 'AP 61',
   'district': 'BROOKLIN PAULISTA',
   'city': 'SAO PAULO',
   'state': 'SP',
   'zipcode': 4562002,
   'latitude': 0,
   'longitude': 0,
   'iptuId': 'd66ada2b-ac48-4be0-a347-0dbe7afbab72',
   'zipCode': 4562002,
   'iptuMatch': True,
   'score': 1}]}

In [67]:
len(lista_contatos)

82765

In [73]:
lista_contatos = [contato for contato in lista_contatos if len(contato.get('document', '')) == 11]

In [74]:
len(lista_contatos)

64878

In [75]:
lista_documentos = list(set([contato['document'] for contato in lista_contatos]))

In [76]:
len(lista_documentos)

5330

In [77]:
lista_contatos[0]

{'document': '82808937849',
 'name': 'Antonio Carlos Bressan',
 'birthDate': '',
 'type': 'PESSOA_FISICA',
 'addresses': [{'street': 'RUA INDIANA',
   'number': 1153,
   'extra': 'AP 61',
   'district': 'BROOKLIN PAULISTA',
   'city': 'SAO PAULO',
   'state': 'SP',
   'zipcode': 4562002,
   'latitude': 0,
   'longitude': 0,
   'iptuId': 'd66ada2b-ac48-4be0-a347-0dbe7afbab72',
   'zipCode': 4562002,
   'iptuMatch': True,
   'score': 1}]}

In [93]:
dicionario_cpf_nome = {contato['document'] : contato['name'].lower() for contato in lista_contatos}

In [94]:
df = pd.read_csv('top_props_bairros_julio.csv', index_col=0)

In [95]:
df.head()

,nome,cpf,quantidade
20234,yara rodrigues,XXXXXX4386XXXX,48
17795,ruggiero marcos di giaimo,XXXXXX2591XXXX,23
12782,maria da conceicao paulino schwittay,XXXXXX2348XXXX,17
192,adriana eduardo daud maluf,XXXXXX9151XXXX,16
5819,estudo rosario catanzaro,XXXXXX1718XXXX,15


In [96]:
df['key_cpf'] = df['cpf'].str.replace('X', '')

In [97]:
df['possiveis_cpfs'] = df['key_cpf'].apply(lambda x: [numero for numero in lista_documentos if x in numero])


In [98]:
df['possiveis_nomes'] = df['possiveis_cpfs'].apply(lambda x: [dicionario_cpf_nome[numero] for numero in x])

In [104]:
df['match_documento'] = df[['nome','possiveis_nomes', 'possiveis_cpfs']].apply(lambda x: x['possiveis_cpfs'][x['possiveis_nomes'].index(x['nome'])] if x['nome'] in x['possiveis_nomes'] else '', axis = 1)

In [105]:
df[df['nome'] == 'estudo rosario catanzaro'].head()

,nome,cpf,quantidade,key_cpf,possiveis_cpfs,possiveis_nomes,match_documento
5819,estudo rosario catanzaro,XXXXXX1718XXXX,15,1718,"[02231718839, 07655171831, 11113171863, 512171...","[adelino marques caldeira filho, andre pellizz...",64417182868


In [106]:
df.to_csv('top_props_bairros_julio_tratado.csv')